# 교육학 연구자를 위한 Computational NLP — 손으로 해보기
**고려대학교 특강 · 참여활동 (Google Colab)**

> 설치·코딩 경험이 없어도 됩니다. 상단 메뉴 **런타임 → 모두 실행**(Runtime → Run all)만 누르면 끝.
> GPU도, API 키도, 회원가입도 필요 없습니다. (정적 임베딩 = CPU만으로 동작)

오늘 확인할 것: **컴퓨터가 '단어를 세는' 게 아니라 '의미를 좌표로' 다룬다**는 것.


## 0단계 · 준비 (약 20초)
임베딩 라이브러리를 설치합니다. 아래 셀의 ▶ 재생 버튼을 누르세요.


In [ ]:
# torch/GPU 불필요한 가벼운 정적 임베딩 라이브러리
!pip -q install model2vec

## 1단계 · 의미를 좌표로 바꾸기
문장을 숫자 벡터(좌표)로 바꾸는 다국어 모델을 불러옵니다.
`encode()` 한 번이면 한국어 문장이 의미 공간의 한 점이 됩니다.


In [ ]:
from model2vec import StaticModel
import numpy as np

# 한국어를 포함한 다국어 정적 임베딩 (약 50MB, 자동 다운로드)
model = StaticModel.from_pretrained('minishlab/potion-multilingual-128M')

def embed(texts):
    v = model.encode(texts)
    return v / np.linalg.norm(v, axis=1, keepdims=True)   # 길이 1로 정규화

def similarity(a, b):
    return float(np.dot(a, b))   # 코사인 유사도 (1에 가까울수록 비슷)

print('준비 완료 — 문장 하나가', model.encode(['안녕하세요']).shape[1], '차원 좌표가 됩니다.')

## 2단계 · 같은 단어, 다른 의미
'죽겠다'와 '죽인다'는 글자가 닮았지만 감정은 정반대입니다.
**단어를 세는 도구는 이 둘을 구분하지 못하지만, 임베딩은 어떨까요?**


In [ ]:
sents = [
    '이 문제 진짜 죽겠다, 너무 어려워서 포기하고 싶어.',     # 좌절
    '와 이거 완전 죽인다, 이렇게 재밌는 실험은 처음이야.',     # 흥분
    '계속 오류가 나서 코드를 한 줄씩 다시 읽어봤다.',         # 성찰/전략
]
E = embed(sents)
print(f'좌절 ↔ 흥분 유사도 : {similarity(E[0], E[1]):.3f}')
print(f'좌절 ↔ 성찰 유사도 : {similarity(E[0], E[2]):.3f}')
print()
print('해석: 표면 글자는 닮았어도(죽겠다/죽인다) 유사도는 낮게 벌어집니다.')
print('     감정은 단어가 아니라 "맥락"에서 나오기 때문입니다.')

## 3단계 · 의미로 검색하기 (키워드가 아니라 뜻)
질문과 **같은 단어가 하나도 없어도**, 뜻이 가까우면 찾아냅니다.
아래는 '자기조절학습'을 검색어로 네 문장을 정렬합니다.


In [ ]:
corpus = [
    '학생이 실수를 스스로 발견하고 방법을 바꿨다.',
    '교사가 따뜻하게 격려하며 다음 단계를 제안했다.',
    '오늘 날씨가 맑아서 소풍을 갔다.',
    '발표가 긴장됐지만 핵심 논지는 분명했다.',
]
C = embed(corpus)
query = embed(['자기 조절 학습, 스스로 전략 수정'])[0]

scores = [similarity(query, c) for c in C]
for i in np.argsort(scores)[::-1]:
    print(f'{scores[i]:+.3f}   {corpus[i]}')
print()
print('해석: "자기조절"이란 단어가 없는 첫 문장이 1위,')
print('     무관한 "소풍" 문장은 꼴찌(음수). = 키워드가 아니라 의미로 찾음.')

## 4단계 · LIWC식 ‘단어 세기’와 비교하기
임베딩 이전의 고전적 방법은 **감정 사전으로 단어를 세는 것**입니다(LIWC 방식).
해석은 쉽지만, 아래처럼 **부정어**에 약합니다.


In [ ]:
neg = {'어렵','힘들','짜증','포기','실패','좌절'}   # 부정 감정 사전(축소판)
pos = {'재미','좋','신기','성공','자신','뿌듯'}   # 긍정 감정 사전

tests = [
    '이 문제 진짜 어려워서 포기하고 싶어.',
    '이 과제 하나도 안 어렵고 재미있었다.',   # 실제로는 긍정!
    '실험이 신기하고 뿌듯했다.',
]
for s in tests:
    n = sum(w in s for w in neg); p = sum(w in s for w in pos)
    verdict = '부정' if n > p else ('긍정' if p > n else '중립')
    print(f'neg={n} pos={p} → {verdict:2s} | {s}')
print()
print('해석: 2번은 "안 어렵고 재미있었다"(긍정)인데 "어렵"이 잡혀 오판 위험.')
print('     단어 세기는 부정·맥락·반어에 약합니다 → 그래서 임베딩/맥락이 필요.')

## 5단계 · 문장들을 2D 지도에 그려보기
임베딩(수백 차원)을 **2차원으로 눌러** 눈으로 봅니다. 가까운 점 = 비슷한 뜻.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

sents6 = [
    '학생이 스스로 오류를 찾아 방법을 바꿨다.',
    '교사가 따뜻하게 격려했다.',
    '오늘 소풍을 갔다 즐거웠다.',
    '발표가 긴장됐지만 해냈다.',
    '실험 데이터를 정리했다.',
    '친구와 점심을 먹었다.',
]
P = PCA(n_components=2).fit_transform(embed(sents6))
plt.figure(figsize=(7,5))
plt.scatter(P[:,0], P[:,1], s=420, c=range(len(sents6)), cmap='tab10')
for i,(x,y) in enumerate(P):
    plt.annotate(str(i+1), (x,y), ha='center', va='center', color='white', fontweight='bold')
plt.title('Sentences in 2D meaning space (PCA)'); plt.xlabel('dim 1'); plt.ylabel('dim 2')
plt.grid(alpha=.3); plt.show()
for i,s in enumerate(sents6): print(i+1, s)

## 6단계 · 감정은 점수가 아니라 ‘궤적’
성찰문을 문장 순서대로 두고, 각 문장이 **좌절/자신감 앵커**와 얼마나 가까운지 그립니다.
좌절이 올랐다가 회복되는 **흐름**이 보이면, 감정은 한 숫자가 아니라 과정임을 뜻합니다.


In [ ]:
traj = [
    '처음엔 정말 짜증나서 포기하고 싶었다.',
    '계속 오류가 나서 답답했다.',
    '한 줄씩 다시 읽어봤다.',
    '원인을 찾았다.',
    '다음엔 더 잘할 수 있겠다.',
]
a_neg = embed(['좌절 답답 포기'])[0]     # 좌절 앵커
a_pos = embed(['자신감 성취 희망'])[0]   # 자신감 앵커
Tt = embed(traj)
xs = range(1, len(traj)+1)
plt.figure(figsize=(7,4))
plt.plot(xs, [similarity(Tt[i], a_neg) for i in range(len(traj))], 'o-', label='frustration')
plt.plot(xs, [similarity(Tt[i], a_pos) for i in range(len(traj))], 's-', label='confidence')
plt.xlabel('sentence #'); plt.ylabel('similarity to anchor'); plt.title('Emotion trajectory')
plt.legend(); plt.grid(alpha=.3); plt.show()
for i,s in enumerate(traj): print(i+1, s)

## 7단계 · 라벨 없이 주제 묶기 (군집화)
질문 글들을 **정답 라벨 없이** 비슷한 것끼리 자동으로 묶습니다(BERTopic의 축소판).


In [ ]:
from sklearn.cluster import KMeans
posts = [
    '시험 범위가 어디까지인가요?',
    '과제 마감이 언제죠?',
    '개념이 이해가 안 돼요 도와주세요',
    '이 부분 원리가 궁금합니다',
    '제출은 어디에 하나요?',
    '왜 이렇게 되는지 설명해주세요',
]
labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(embed(posts))
for c in sorted(set(labels)):
    print(f'[군집 {c}]')
    for i,s in enumerate(posts):
        if labels[i]==c: print('   -', s)
print()
print('해석: 라벨을 주지 않아도 "학습 질문"과 "행정 질문"이 갈립니다.')
print('     단, 군집에 이름을 붙이는 것(=해석)은 사람의 몫입니다.')

## 8단계 · 직접 바꿔보기 ✍️
아래 문장들을 **여러분의 연구 주제**로 바꿔 다시 실행해 보세요.
(예: 학생 성찰문, 토론 글, 교사 피드백 — 단, 실제 학생 데이터 대신 예시/합성 문장으로.)


In [ ]:
my_query = '여기에 내 연구 관심을 한 문장으로'
my_texts = [
    '바꿔볼 문장 1',
    '바꿔볼 문장 2',
    '바꿔볼 문장 3',
]
Cq = embed(my_texts)
qv = embed([my_query])[0]
for i in np.argsort([similarity(qv, c) for c in Cq])[::-1]:
    print(f'{similarity(qv, Cq[i]):+.3f}   {my_texts[i]}')

---
## (선택) · LLM에게 공감 코딩 시키기
임베딩은 '의미의 거리'를 재고, LLM은 '라벨과 근거'를 답니다.
아래는 **본인 API 키가 있을 때만** 실행하는 선택 활동입니다. 키가 없으면 폰의 ChatGPT/Claude 앱에 같은 프롬프트를 붙여넣어도 동일합니다.

> ⚠️ **실제 학생 데이터는 외부 모델에 넣지 마세요.** 합성/공개 예시만.


In [ ]:
PROMPT = '''다음 교사 피드백을 공감 코딩해줘. 각 문장을
[정서적 인정 / 관점 수용 / 실행 가능한 조언 / 과잉 도움] 중 하나로 분류하고,
근거를 한 줄씩 붙여줘. 분류가 애매하면 '불확실'로 표시해.

피드백: "이번 발표 준비하느라 정말 고생 많았어요. 긴장한 게 느껴졌지만
핵심 논지는 분명했어요. 다음엔 도입을 30초로 줄이고 예시를 하나만 남겨보면 어떨까요?"'''

print(PROMPT)
print()
print('↑ 이 프롬프트를 ChatGPT/Claude 앱에 붙여넣으세요.')
print('  (API 키가 있으면 아래 주석을 풀어 코드로도 호출할 수 있습니다.)')

# --- 선택: OpenAI API 키가 있을 때 ---
# !pip -q install openai
# from openai import OpenAI
# client = OpenAI(api_key='sk-...여기에_본인_키...')
# r = client.chat.completions.create(model='gpt-4o-mini',
#         messages=[{'role':'user','content':PROMPT}])
# print(r.choices[0].message.content)

---
### 오늘 손으로 확인한 것
1. 임베딩은 문장을 **의미 좌표**로 바꾼다 (단어 세기가 아님).
2. **거리 = 의미 유사도** — 같은 단어라도 맥락이 다르면 멀어진다.
3. **LIWC식 단어 세기**는 쉽지만 부정·맥락에 약하다 → 맥락 임베딩이 보완.
4. 임베딩을 **2D로** 보면 문장 간 관계가 눈에 보인다.
5. 감정은 한 점수가 아니라 **궤적**(오르내림)으로 본다.
6. **군집화**는 라벨 없이 주제를 묶지만, 이름 붙이기(해석)는 사람의 몫.
7. LLM은 **라벨과 근거의 초안**을 준다 — 신뢰도·과잉주장 검증은 연구자의 몫.

**construct-first**: 도구가 아니라 '무엇을, 어떤 증거로, 어디까지'가 먼저입니다.
